# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# UNI prediction-contributing region visualization

This notebook visualizes a saved grouped out-of-fold UNI explanation without retraining the model. It shows the original histology tile, UNI gradient-weighted rollout, the top-ranked patches, and matched input-occlusion evidence. Highlighted regions indicate contribution to the model score, not biological causality.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image as DisplayImage, display


def locate_project_root():
    return _PUBLICATION_ROOT


# PROJECT_ROOT = locate_project_root()
PROJECT_ROOT = Path('.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Methods.FinalComparison.region_example import save_uni_region_example

GROUPED_DIR = PROJECT_ROOT / "artifacts" / "grouped_oof_faithfulness"
THREE_MODEL_DIR = PROJECT_ROOT / "artifacts" / "dinov2_three_model"
OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "final_three_model_comparison"
    / "figures"
    / "prediction_region_examples"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)

## Configuration

The default is a fixed, correctly classified tumor example. Set `COHORT_ID = None` to select the correct image with the highest attribution-occlusion Spearman correlation within `CLASS_NAME`. Automatic selection is for illustration only and must not be used as a quantitative estimate of overall faithfulness.

In [ ]:
# Fixed example used in the project visualization.
COHORT_ID = "39f8deff00e87441"

# Used only when COHORT_ID is None or unavailable. Use None for any class.
CLASS_NAME = "01_TUMOR"
REQUIRE_CORRECT = True

# For an incorrect image, use "predicted" to explain the wrong prediction
# or "true" to explain the score assigned to the ground-truth class.
TARGET_ROLE = "predicted"
TOP_FRACTION = 0.10

assert TARGET_ROLE in {"predicted", "true"}
assert 0 < TOP_FRACTION <= 1

In [ ]:
manifest = pd.read_csv(GROUPED_DIR / "faithfulness_cohort_manifest.csv")
metrics = pd.read_csv(
    THREE_MODEL_DIR / "faithfulness" / "three_model_faithfulness_metrics.csv"
)

eligible = metrics.loc[
    metrics["model"].eq("UNI")
    & metrics["method"].eq("gradient_attention_rollout")
    & metrics["target_role"].eq(TARGET_ROLE)
].copy()
if CLASS_NAME is not None:
    eligible = eligible.loc[eligible["class_name"].eq(CLASS_NAME)]
if REQUIRE_CORRECT:
    eligible = eligible.loc[eligible["correct"].astype(bool)]

available_ids = set(eligible["cohort_id"].astype(str))
if COHORT_ID is not None and str(COHORT_ID) in available_ids:
    selected_id = str(COHORT_ID)
    selection_note = "fixed cohort ID"
else:
    if eligible.empty:
        raise ValueError("No image matches the current class, correctness, and target-role filters")
    selected_id = str(
        eligible.sort_values("attribution_occlusion_spearman", ascending=False)
        .iloc[0]["cohort_id"]
    )
    selection_note = "automatic illustrative selection by highest Spearman correlation"

selected_manifest = manifest.loc[
    manifest["cohort_id"].astype(str).eq(selected_id)
].iloc[0]
selected_metric = metrics.loc[
    metrics["cohort_id"].astype(str).eq(selected_id)
    & metrics["model"].eq("UNI")
    & metrics["method"].eq("gradient_attention_rollout")
    & metrics["target_role"].eq(TARGET_ROLE)
].iloc[0]

display(
    pd.DataFrame(
        [
            {
                "selection": selection_note,
                "cohort_id": selected_id,
                "source_case": selected_manifest["case_id"],
                "true_class": selected_manifest["class_name"],
                "UNI_prediction": selected_manifest["uni_predicted_class_name"],
                "UNI_confidence": selected_manifest["uni_confidence"],
                "correct": bool(selected_metric["correct"]),
                "target_role": TARGET_ROLE,
                "attribution_occlusion_spearman": selected_metric["attribution_occlusion_spearman"],
                "top_minus_random_logit_auc": selected_metric["top_minus_random_target_logit_auc"],
                "top_minus_random_margin_auc": selected_metric["top_minus_random_margin_auc"],
            }
        ]
    ).style.format(precision=4)
)

In [ ]:
output_path = (
    OUTPUT_DIR
    / f"{selected_id}_uni_{TARGET_ROLE}_top_{TOP_FRACTION * 100:.0f}pct.png"
)
save_uni_region_example(
    PROJECT_ROOT,
    selected_id,
    output_path,
    target_role=TARGET_ROLE,
    top_fraction=TOP_FRACTION,
)
display(DisplayImage(filename=str(output_path)))
print("Saved:", output_path)

## Class-complete gallery

This section creates additional examples across all eight tissue classes. By default, it chooses one correctly classified image whose attribution-occlusion correlation is closest to the median for its class. This provides a representative visualization rather than selecting only the most favorable explanation. Set `GALLERY_CORRECTNESS = "incorrect"` to inspect errors.

In [ ]:
RUN_CLASS_GALLERY = True
GALLERY_CORRECTNESS = "correct"  # "correct", "incorrect", or "all"
GALLERY_TARGET_ROLE = "predicted"  # use "true" for true-class error analysis
GALLERY_EXAMPLES_PER_CLASS = 1
GALLERY_TOP_FRACTION = 0.10

assert GALLERY_CORRECTNESS in {"correct", "incorrect", "all"}
assert GALLERY_TARGET_ROLE in {"predicted", "true"}
assert GALLERY_EXAMPLES_PER_CLASS >= 1
assert 0 < GALLERY_TOP_FRACTION <= 1

In [ ]:
if RUN_CLASS_GALLERY:
    gallery_pool = metrics.loc[
        metrics["model"].eq("UNI")
        & metrics["method"].eq("gradient_attention_rollout")
        & metrics["target_role"].eq(GALLERY_TARGET_ROLE)
    ].copy()
    if GALLERY_CORRECTNESS == "correct":
        gallery_pool = gallery_pool.loc[gallery_pool["correct"].astype(bool)]
    elif GALLERY_CORRECTNESS == "incorrect":
        gallery_pool = gallery_pool.loc[~gallery_pool["correct"].astype(bool)]

    selected_groups = []
    for class_name, class_rows in gallery_pool.groupby("class_name", sort=True):
        class_rows = class_rows.copy()
        class_median = class_rows["attribution_occlusion_spearman"].median()
        class_rows["distance_from_class_median"] = (
            class_rows["attribution_occlusion_spearman"] - class_median
        ).abs()
        selected_groups.append(
            class_rows.sort_values(
                ["distance_from_class_median", "cohort_id"]
            ).head(GALLERY_EXAMPLES_PER_CLASS)
        )

    if not selected_groups:
        raise ValueError("No images match the gallery configuration")
    gallery_selection = pd.concat(selected_groups, ignore_index=True)
    gallery_selection = gallery_selection.merge(
        manifest[["cohort_id", "case_id", "uni_predicted_class_name", "uni_confidence"]],
        on="cohort_id",
        how="left",
        suffixes=("", "_manifest"),
    )

    display(
        gallery_selection[
            [
                "class_name",
                "cohort_id",
                "case_id",
                "uni_predicted_class_name",
                "uni_confidence",
                "correct",
                "attribution_occlusion_spearman",
                "top_minus_random_target_logit_auc",
            ]
        ].style.format(precision=4)
    )

    gallery_dir = OUTPUT_DIR / f"gallery_{GALLERY_CORRECTNESS}_{GALLERY_TARGET_ROLE}"
    gallery_dir.mkdir(parents=True, exist_ok=True)
    for row in gallery_selection.itertuples(index=False):
        gallery_path = (
            gallery_dir
            / f"{row.class_name}_{row.cohort_id}_top_{GALLERY_TOP_FRACTION * 100:.0f}pct.png"
        )
        save_uni_region_example(
            PROJECT_ROOT,
            str(row.cohort_id),
            gallery_path,
            target_role=GALLERY_TARGET_ROLE,
            top_fraction=GALLERY_TOP_FRACTION,
        )
        display(DisplayImage(filename=str(gallery_path), width=1400))
    print(f"Saved {len(gallery_selection)} gallery examples to: {gallery_dir}")

## Reading the figure

- Warm colors in gradient-weighted rollout indicate patches with larger attribution scores for the selected target class.
- Outlined boxes identify the top fraction of attributed patches on the shared 14 x 14 grid.
- Warm colors in input occlusion indicate patches whose removal reduces the selected target-class logit more strongly.
- Agreement between attribution and occlusion is summarized by the per-image Spearman correlation.
- These maps describe model behavior only; they do not establish biological or pathological causality.